# Notebook 08: `/URI` Counterfactual Sensitivity, Ablation & Domain-Shift Audit

This notebook evaluates how strongly DocSentry's shipped XGBoost classifier depends on link annotations (`strings_regex_/URI`) and regex indicators overall, and checks where the synthetic and user evaluation PDFs sit relative to the training benign distribution.

- **Seed**: `42` throughout.
- **No shipped model is modified**: All ablation models are saved strictly under `models/ablation/`.
- **Privacy**: Personal PDFs are referenced strictly as `user_pdf_1` ... `user_pdf_N` with no real filenames, hashes, or raw feature vectors saved.

In [1]:
import sys
import json
import time
from pathlib import Path

import joblib
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

np.random.seed(42)

# Project root resolution
ROOT = Path('..').resolve() if Path('../data/processed').exists() else Path('.').resolve()
sys.path.insert(0, str(ROOT))

PROCESSED_DIR = ROOT / 'data/processed'
MODELS_DIR = ROOT / 'models'
ABLATION_DIR = MODELS_DIR / 'ablation'
ABLATION_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = ROOT / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Load schemas, splits, and shipped artifacts
feature_names = json.loads((PROCESSED_DIR / 'feature_names.json').read_text(encoding='utf-8'))
thresholds = json.loads((MODELS_DIR / 'thresholds.json').read_text(encoding='utf-8'))
best_params = json.loads((REPORTS_DIR / 'best_params.json').read_text(encoding='utf-8'))

X_tr = pd.read_parquet(PROCESSED_DIR / 'X_train_tree.parquet')
y_tr = pd.read_parquet(PROCESSED_DIR / 'y_train.parquet')['label'].values
X_va = pd.read_parquet(PROCESSED_DIR / 'X_val_tree.parquet')
y_va = pd.read_parquet(PROCESSED_DIR / 'y_val.parquet')['label'].values
X_te = pd.read_parquet(PROCESSED_DIR / 'X_test_tree.parquet')
y_te = pd.read_parquet(PROCESSED_DIR / 'y_test.parquet')['label'].values

model = joblib.load(MODELS_DIR / 'best_model.joblib')
calibrator = joblib.load(MODELS_DIR / 'calibrator.joblib')

t_op = float(thresholds['optimal_decision_threshold'])
band_low = float(thresholds['review_band']['low'])
band_high = float(thresholds['review_band']['high'])

print(f"Loaded splits: Train={X_tr.shape}, Val={X_va.shape}, Test={X_te.shape}")
print(f"Operating threshold t_op = {t_op:.4f}, Review band = [{band_low:.3f}, {band_high:.3f}]")

<PROJECT_ROOT>\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded splits: Train=(42000, 658), Val=(9999, 658), Test=(12000, 658)
Operating threshold t_op = 0.4083, Review band = [0.290, 0.550]


## A1. Baseline Reproduction Check

In [2]:
def score_calibrated(X_df):
    raw_p = model.predict_proba(X_df)[:, 1]
    cal_p = calibrator.predict(raw_p)
    preds = (cal_p >= t_op).astype(int)
    verdicts = np.where(cal_p < band_low, 'benign', np.where(cal_p <= band_high, 'needs_review', 'malicious'))
    return raw_p, cal_p, preds, verdicts

raw_base, cal_base, pred_base, verd_base = score_calibrated(X_te)
base_errors = int(np.sum(pred_base != y_te))
uri_col = 'strings_regex_/URI'
mal_uri0_mask = (y_te == 1) & (X_te[uri_col] == 0)
mal_uri0_rec = float(np.mean(pred_base[mal_uri0_mask] == 1))

print(f"Baseline Test Errors at t={t_op:.4f}: {base_errors} (expected 554)")
print(f"Baseline /URI==0 Malicious Recall (calibrated): {mal_uri0_rec*100:.2f}% (expected 71.32%)")

assert base_errors == 554, f"Expected 554 test errors, got {base_errors}"
assert abs(mal_uri0_rec - 0.7132146204311153) < 1e-4, f"Unexpected /URI==0 recall: {mal_uri0_rec:.4f}"
print("Baseline assertions PASSED.")

Baseline Test Errors at t=0.4083: 554 (expected 554)
Baseline /URI==0 Malicious Recall (calibrated): 71.32% (expected 71.32%)
Baseline assertions PASSED.


## A2 & A3. Counterfactual Link Removal (`/URI = 0`) and Addition (`/URI = 1, 5, 25`)

In [3]:
LABELS_3 = ['benign', 'needs_review', 'malicious']

def transition_matrix(v_before, v_after):
    mat = pd.DataFrame(0, index=LABELS_3, columns=LABELS_3)
    for b, a in zip(v_before, v_after):
        mat.loc[b, a] += 1
    return mat

def paired_bootstrap_delta(y_true, p_before, p_after, n_boot=1000, seed=42):
    rng = np.random.RandomState(seed)
    n = len(y_true)
    d_rec, d_fpr = [], []
    for _ in range(n_boot):
        idx = rng.randint(0, n, size=n)
        yb = y_true[idx]
        pb = p_before[idx]
        pa = p_after[idx]
        mal = (yb == 1)
        ben = (yb == 0)
        d_rec.append(np.mean(pa[mal] == 1) - np.mean(pb[mal] == 1))
        d_fpr.append(np.mean(pa[ben] == 1) - np.mean(pb[ben] == 1))
    return (
        float(np.mean(d_rec)), float(np.percentile(d_rec, 2.5)), float(np.percentile(d_rec, 97.5)),
        float(np.mean(d_fpr)), float(np.percentile(d_fpr, 2.5)), float(np.percentile(d_fpr, 97.5)),
    )

def run_counterfactual(uri_value, exp_name):
    X_cf = X_te.copy()
    X_cf[uri_col] = float(uri_value)
    _, cal_cf, pred_cf, verd_cf = score_calibrated(X_cf)

    mal = (y_te == 1)
    ben = (y_te == 0)
    rev_before = float(np.mean(verd_base == 'needs_review'))
    rev_after = float(np.mean(verd_cf == 'needs_review'))

    rec_before = float(np.mean(pred_base[mal] == 1))
    rec_after = float(np.mean(pred_cf[mal] == 1))
    fpr_before = float(np.mean(pred_base[ben] == 1))
    fpr_after = float(np.mean(pred_cf[ben] == 1))

    dr_m, dr_lo, dr_hi, df_m, df_lo, df_hi = paired_bootstrap_delta(y_te, pred_base, pred_cf)

    tmat_all = transition_matrix(verd_base, verd_cf)
    tmat_mal = transition_matrix(verd_base[mal], verd_cf[mal])
    tmat_ben = transition_matrix(verd_base[ben], verd_cf[ben])

    res = {
        'experiment': exp_name,
        'uri_value': uri_value,
        'mal_mean_cal_p_before': round(float(np.mean(cal_base[mal])), 4),
        'mal_mean_cal_p_after': round(float(np.mean(cal_cf[mal])), 4),
        'ben_mean_cal_p_before': round(float(np.mean(cal_base[ben])), 4),
        'ben_mean_cal_p_after': round(float(np.mean(cal_cf[ben])), 4),
        'recall_before': round(rec_before, 4),
        'recall_after': round(rec_after, 4),
        'recall_delta_ci95': [round(dr_lo, 4), round(dr_hi, 4)],
        'fpr_before': round(fpr_before, 4),
        'fpr_after': round(fpr_after, 4),
        'fpr_delta_ci95': [round(df_lo, 4), round(df_hi, 4)],
        'review_rate_before': round(rev_before, 4),
        'review_rate_after': round(rev_after, 4),
        'transition_all': tmat_all.to_dict(),
        'transition_malicious': tmat_mal.to_dict(),
        'transition_benign': tmat_ben.to_dict(),
    }

    if uri_value == 0:
        already_0 = (X_te[uri_col] == 0)
        gt_0 = (X_te[uri_col] > 0)
        assert np.all(verd_base[already_0] == verd_cf[already_0]), "Files with /URI==0 changed!"
        res['already_zero_unchanged_count'] = int(np.sum(already_0))
        res['gt_zero_mal_count'] = int(np.sum(gt_0 & mal))
        res['gt_zero_mal_recall_before'] = round(float(np.mean(pred_base[gt_0 & mal] == 1)), 4)
        res['gt_zero_mal_recall_after'] = round(float(np.mean(pred_cf[gt_0 & mal] == 1)), 4)
        res['gt_zero_mal_flipped_from_malicious'] = int(np.sum((verd_base[gt_0 & mal] == 'malicious') & (verd_cf[gt_0 & mal] != 'malicious')))
        res['gt_zero_mal_flipped_to_benign'] = int(np.sum((verd_base[gt_0 & mal] == 'malicious') & (verd_cf[gt_0 & mal] == 'benign')))

    print(f"=== {exp_name} (/URI = {uri_value}) ===")
    print(f"  Malicious: mean cal P {res['mal_mean_cal_p_before']:.4f} -> {res['mal_mean_cal_p_after']:.4f} | Recall {rec_before*100:.2f}% -> {rec_after*100:.2f}% (delta 95% CI [{dr_lo*100:+.2f}%, {dr_hi*100:+.2f}%])")
    print(f"  Benign:    mean cal P {res['ben_mean_cal_p_before']:.4f} -> {res['ben_mean_cal_p_after']:.4f} | FPR    {fpr_before*100:.2f}% -> {fpr_after*100:.2f}% (delta 95% CI [{df_lo*100:+.2f}%, {df_hi*100:+.2f}%])")
    print(f"  Review Rate: {rev_before*100:.2f}% -> {rev_after*100:.2f}%")
    print("  Transition Matrix (All Test):")
    print(tmat_all)
    if uri_value == 0:
        print(f"  Already /URI==0: {res['already_zero_unchanged_count']} files strictly unchanged.")
        print(f"  /URI>0 Malicious ({res['gt_zero_mal_count']}): Recall {res['gt_zero_mal_recall_before']*100:.2f}% -> {res['gt_zero_mal_recall_after']*100:.2f}%, flipped out of malicious={res['gt_zero_mal_flipped_from_malicious']} (to benign={res['gt_zero_mal_flipped_to_benign']})")
    print()
    return res

cf_results = [
    run_counterfactual(0, 'A2_remove_links_uri_0'),
    run_counterfactual(1, 'A3_add_links_uri_1'),
    run_counterfactual(5, 'A3_add_links_uri_5'),
    run_counterfactual(25, 'A3_add_links_uri_25'),
]

=== A2_remove_links_uri_0 (/URI = 0) ===
  Malicious: mean cal P 0.9261 -> 0.6728 | Recall 92.73% -> 63.45% (delta 95% CI [-30.51%, -28.13%])
  Benign:    mean cal P 0.0364 -> 0.0219 | FPR    1.97% -> 0.82% (delta 95% CI [-1.40%, -0.90%])
  Review Rate: 3.77% -> 16.02%
  Transition Matrix (All Test):
              benign  needs_review  malicious
benign          5906             0          0
needs_review     166           286          0
malicious        238          1636       3768
  Already /URI==0: 5792 files strictly unchanged.
  /URI>0 Malicious (4933): Recall 97.36% -> 61.75%, flipped out of malicious=1814 (to benign=195)



=== A3_add_links_uri_1 (/URI = 1) ===
  Malicious: mean cal P 0.9261 -> 0.9569 | Recall 92.73% -> 95.98% (delta 95% CI [+2.81%, +3.71%])
  Benign:    mean cal P 0.0364 -> 0.1679 | FPR    1.97% -> 12.60% (delta 95% CI [+9.90%, +11.41%])
  Review Rate: 3.77% -> 10.02%
  Transition Matrix (All Test):
              benign  needs_review  malicious
benign          4380          1018        508
needs_review       6           179        267
malicious          0             5       5637



=== A3_add_links_uri_5 (/URI = 5) ===
  Malicious: mean cal P 0.9261 -> 0.9553 | Recall 92.73% -> 95.78% (delta 95% CI [+2.61%, +3.52%])
  Benign:    mean cal P 0.0364 -> 0.1614 | FPR    1.97% -> 12.13% (delta 95% CI [+9.46%, +10.95%])
  Review Rate: 3.77% -> 9.84%
  Transition Matrix (All Test):
              benign  needs_review  malicious
benign          4436           984        486
needs_review       7           189        256
malicious          0             8       5634



=== A3_add_links_uri_25 (/URI = 25) ===
  Malicious: mean cal P 0.9261 -> 0.9556 | Recall 92.73% -> 95.92% (delta 95% CI [+2.73%, +3.64%])
  Benign:    mean cal P 0.0364 -> 0.1787 | FPR    1.97% -> 13.40% (delta 95% CI [+10.67%, +12.22%])
  Review Rate: 3.77% -> 10.54%
  Transition Matrix (All Test):
              benign  needs_review  malicious
benign          4272          1089        545
needs_review      23           163        266
malicious          0            13       5629



## A4. Dose-Response & Individual Conditional Expectation (ICE) Curves

In [4]:
rng = np.random.RandomState(42)
idx_ben = rng.choice(np.where(y_te == 0)[0], size=500, replace=False)
idx_mal = rng.choice(np.where(y_te == 1)[0], size=500, replace=False)

uri_grid = [0, 1, 2, 3, 5, 8, 12, 20, 30, 50]

def sweep_uri(indices):
    X_sub = X_te.iloc[indices].copy()
    curves = np.zeros((len(indices), len(uri_grid)))
    for j, u in enumerate(uri_grid):
        X_sub[uri_col] = float(u)
        curves[:, j] = calibrator.predict(model.predict_proba(X_sub)[:, 1])
    return curves

curves_ben = sweep_uri(idx_ben)
curves_mal = sweep_uri(idx_mal)

# Figure 40: Mean + 10th-90th percentile band (Verdict colors, NO blue)
fig, ax = plt.subplots(figsize=(9.5, 5.5))
x_idx = np.arange(len(uri_grid))

for curves, label, color in [
    (curves_mal, 'Malicious (n=500)', '#C0392B'),
    (curves_ben, 'Benign (n=500)', '#1F7A45'),
]:
    mean_c = np.mean(curves, axis=0)
    p10 = np.percentile(curves, 10, axis=0)
    p90 = np.percentile(curves, 90, axis=0)
    ax.plot(x_idx, mean_c, marker='o', linewidth=2.2, color=color, label=label)
    ax.fill_between(x_idx, p10, p90, color=color, alpha=0.16)

ax.axhspan(band_low, band_high, color='#A86B00', alpha=0.18, label=f'Review Band [{band_low:.2f}, {band_high:.2f}]')
ax.axhline(t_op, color='#141412', linestyle='--', linewidth=1.6, label=f'Operating Threshold ({t_op:.4f})')

ax.set_xticks(x_idx)
ax.set_xticklabels([str(u) for u in uri_grid])
ax.set_xlabel('Injected strings_regex_/URI Count', fontsize=11, fontweight='bold')
ax.set_ylabel('Calibrated Malware Probability', fontsize=11, fontweight='bold')
ax.set_title('Figure 40: /URI Counterfactual Dose-Response (Mean & 10th–90th Percentile Band)', fontsize=12, fontweight='bold')
ax.set_ylim(-0.02, 1.04)
ax.grid(True, linestyle=':', alpha=0.6)
ax.legend(loc='best', frameon=True)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '40_uri_dose_response.png', dpi=200)
plt.close()

# Figure 41: 30 Individual ICE curves per class
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, curves, title, color in [
    (axes[0], curves_ben[:30], 'Benign Sample (30 ICE Curves)', '#1F7A45'),
    (axes[1], curves_mal[:30], 'Malicious Sample (30 ICE Curves)', '#C0392B'),
]:
    for row in curves:
        ax.plot(x_idx, row, color=color, alpha=0.35, linewidth=1.2)
    ax.axhspan(band_low, band_high, color='#A86B00', alpha=0.16)
    ax.axhline(t_op, color='#141412', linestyle='--', linewidth=1.5)
    ax.set_xticks(x_idx)
    ax.set_xticklabels([str(u) for u in uri_grid])
    ax.set_xlabel('Injected strings_regex_/URI Count', fontsize=10, fontweight='bold')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(True, linestyle=':', alpha=0.6)

axes[0].set_ylabel('Calibrated Malware Probability', fontsize=10, fontweight='bold')
plt.suptitle('Figure 41: Individual Conditional Expectation (ICE) Curves Under /URI Sweep', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '41_uri_ice_curves.png', dpi=200)
plt.close()
print("Saved Figures 40 and 41.")

Saved Figures 40 and 41.


## A5. Training `/URI` Distribution & Spearman Correlation Context

In [5]:
for cls_name, cls_val in [('Train Benign', 0), ('Train Malicious', 1)]:
    s = X_tr.loc[y_tr == cls_val, uri_col]
    zero_pct = float(np.mean(s == 0) * 100)
    q = np.percentile(s, [0, 25, 50, 75, 90, 99])
    print(f"{cls_name} /URI: zero={zero_pct:.2f}%, quantiles [0,25,50,75,90,99]={np.round(q, 2).tolist()}")

# Top 10 features by mean |SHAP| + general_size
bg = X_tr.iloc[rng.choice(len(X_tr), size=100, replace=False)]
samp_idx = np.concatenate([idx_ben, idx_mal])
explainer = shap.TreeExplainer(model, data=bg)
sv = explainer(X_te.iloc[samp_idx]).values
mean_abs = np.mean(np.abs(sv), axis=0)
top10_idx = np.argsort(mean_abs)[::-1][:10]
corr_cols = [feature_names[i] for i in top10_idx]
if 'general_size' not in corr_cols:
    corr_cols.append('general_size')

print("\nSpearman correlation with strings_regex_/URI on Training Set:")
spearman_rows = []
for c in corr_cols:
    rho, pval = spearmanr(X_tr[uri_col], X_tr[c])
    spearman_rows.append({'feature': c, 'spearman_rho': round(float(rho), 4)})
    print(f"  {c:<35} rho = {rho:+.4f}")

Train Benign /URI: zero=82.84%, quantiles [0,25,50,75,90,99]=[0.0, 0.0, 0.0, 0.0, 4.0, 190.02]
Train Malicious /URI: zero=7.27%, quantiles [0,25,50,75,90,99]=[0.0, 4.0, 22.0, 36.0, 48.0, 76.0]



Spearman correlation with strings_regex_/URI on Training Set:
  strings_regex_/URI                  rho = +1.0000
  strings_regex_https://              rho = +0.8399
  strings_printables                  rho = -0.0549
  strings_numstrings                  rho = -0.1600
  strings_printabledist_68            rho = +0.5504
  strings_regex_decode                rho = -0.2245
  strings_printabledist_80            rho = +0.6741
  strings_printabledist_38            rho = -0.5649
  general_size                        rho = -0.2597
  strings_regex_/FlateDecode          rho = -0.1559


> **Counterfactual Validity Note**: Setting only `strings_regex_/URI` creates feature vectors that real PDFs may not produce (for example, changing `/URI` without updating `general_size`, `strings_numstrings`, `https://`, or printable-character histograms), so these counterfactual results measure the model's isolated sensitivity to the `/URI` feature rather than realistic real-world evasion rates.

## A6. Retrained Ablations (All 658 vs. No `/URI` vs. No Regex)

In [6]:
def eval_ablation(clf, X_v, y_v, X_t, y_t, variant_name):
    def metrics_at_05(X_d, y_d):
        p = clf.predict_proba(X_d)[:, 1]
        pred = (p >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_d, pred).ravel()
        return {
            'f1': round(float(f1_score(y_d, pred)), 4),
            'precision': round(float(precision_score(y_d, pred)), 4),
            'recall': round(float(recall_score(y_d, pred)), 4),
            'fpr': round(float(fp / (fp + tn)), 4),
            'roc_auc': round(float(roc_auc_score(y_d, p)), 4),
            'pr_auc': round(float(average_precision_score(y_d, p)), 4),
        }

    val_m = metrics_at_05(X_v, y_v)
    test_m = metrics_at_05(X_t, y_t)

    # Bootstrap 95% CIs on test (1,000 resamples)
    rng_b = np.random.RandomState(42)
    p_t = clf.predict_proba(X_t)[:, 1]
    pred_t = (p_t >= 0.5).astype(int)
    n_t = len(y_t)
    f1s, recs, fprs = [], [], []
    for _ in range(1000):
        idx = rng_b.randint(0, n_t, size=n_t)
        yb, pb = y_t[idx], pred_t[idx]
        f1s.append(f1_score(yb, pb))
        recs.append(recall_score(yb, pb))
        tn, fp, fn, tp = confusion_matrix(yb, pb).ravel()
        fprs.append(fp / (fp + tn))

    return {
        'variant': variant_name,
        'n_features': X_t.shape[1],
        'val': val_m,
        'test': test_m,
        'test_ci95': {
            'f1': [round(float(np.percentile(f1s, 2.5)), 4), round(float(np.percentile(f1s, 97.5)), 4)],
            'recall': [round(float(np.percentile(recs, 2.5)), 4), round(float(np.percentile(recs, 97.5)), 4)],
            'fpr': [round(float(np.percentile(fprs, 2.5)), 4), round(float(np.percentile(fprs, 97.5)), 4)],
        }
    }

xgb_params = dict(best_params['XGBoost'])
regex_cols = [c for c in feature_names if c.startswith('strings_regex_')]
no_uri_cols = [c for c in feature_names if c != uri_col]
no_regex_cols = [c for c in feature_names if c not in regex_cols]

ablation_specs = [
    ('control_658', feature_names, 'xgb_control_658.joblib'),
    ('no_uri_657', no_uri_cols, 'xgb_no_uri_657.joblib'),
    ('no_regex_581', no_regex_cols, 'xgb_no_regex_581.joblib'),
]

ablation_results = []
for vname, cols, fname in ablation_specs:
    clf = XGBClassifier(**xgb_params)
    clf.fit(X_tr[cols], y_tr)
    joblib.dump(clf, ABLATION_DIR / fname)
    res = eval_ablation(clf, X_va[cols], y_va, X_te[cols], y_te, vname)
    ablation_results.append(res)
    print(f"[{vname}] ({len(cols)} feats) | Val F1={res['val']['f1']:.4f} | Test F1={res['test']['f1']:.4f} {res['test_ci95']['f1']} | Recall={res['test']['recall']:.4f} | FPR={res['test']['fpr']:.4f} | ROC-AUC={res['test']['roc_auc']:.4f}")

[control_658] (658 feats) | Val F1=0.9824 | Test F1=0.9451 [0.941, 0.9494] | Recall=0.9042 | FPR=0.0092 | ROC-AUC=0.9887


[no_uri_657] (657 feats) | Val F1=0.9784 | Test F1=0.9476 [0.9436, 0.9516] | Recall=0.9082 | FPR=0.0087 | ROC-AUC=0.9870


[no_regex_581] (618 feats) | Val F1=0.9573 | Test F1=0.9388 [0.9343, 0.9433] | Recall=0.8950 | FPR=0.0117 | ROC-AUC=0.9842


## A7. Save Counterfactual & Ablation Reports + What This Shows

In [7]:
# Re-build What this shows strictly from computed variables with direction asserts
cf_report = json.loads((REPORTS_DIR / 'uri_counterfactual.json').read_text(encoding='utf-8'))
cf0 = cf_report['counterfactuals'][0]
ab_ctrl, ab_nouri, ab_noreg = cf_report['ablations']

tmat_mal = cf0['transition_malicious']
n_mal_to_rev = int(tmat_mal['needs_review']['malicious'])
n_mal_to_ben = int(tmat_mal['benign']['malicious'])
n_mal_lost_verdict = n_mal_to_rev + n_mal_to_ben
assert n_mal_lost_verdict == cf0['gt_zero_mal_flipped_from_malicious'], 'Transition matrix mismatch!'

rec_before = cf0['recall_before'] * 100
rec_after = cf0['recall_after'] * 100
rec_ci = cf0['recall_delta_ci95']
rev_before = cf0['review_rate_before'] * 100
rev_after = cf0['review_rate_after'] * 100

f1_ctrl = ab_ctrl['test']['f1']
f1_nouri = ab_nouri['test']['f1']
f1_noreg = ab_noreg['test']['f1']

assert rec_after < rec_before, 'Expected recall fall!'
assert rev_after > rev_before, 'Expected review rate rise!'
assert f1_nouri >= f1_ctrl, 'Expected no F1 drop without /URI!'
assert f1_noreg < f1_ctrl, 'Expected F1 loss without all regex!'

s_a = (
    f"(a) Shipped model with links removed (/URI = 0 in the feature table): recall at t = 0.4083 falls "
    f"from {rec_before:.2f}% to {rec_after:.2f}% (paired 95% CI for change: [{rec_ci[0]*100:+.2f}%, {rec_ci[1]*100:+.2f}%]) "
    f"and the review rate rises from {rev_before:.2f}% to {rev_after:.2f}%. Among link-bearing malicious files, "
    f"{n_mal_lost_verdict} lose the malicious verdict: {n_mal_to_rev} move to needs_review and {n_mal_to_ben} move to benign."
)
s_b = (
    "(b) Definition distinction: 'recall at t = 0.4083' counts files with calibrated probability >= 0.4083, "
    "whereas a 'malicious verdict' requires calibrated probability > 0.550 (above the review band). These are different "
    f"quantities, so the {n_mal_lost_verdict} malicious-verdict losses and the {rec_before - rec_after:.2f}-point recall fall at 0.4083 are not the same quantity."
)
s_c = (
    f"(c) Retraining without /URI (657 features): test F1 is {f1_nouri:.4f} (95% CI {ab_nouri['test_ci95']['f1']}) "
    f"vs {f1_ctrl:.4f} (95% CI {ab_ctrl['test_ci95']['f1']}) for the 658-feature control—the intervals overlap, so there is "
    f"no loss in test F1 (validation F1 is {ab_nouri['val']['f1']:.4f} vs {ab_ctrl['val']['f1']:.4f}). This indicates link "
    "information is partly redundant with other features in this dataset (an untested hypothesis is that byte or character "
    "frequencies of '/', 'U', 'R', 'I' or URL schemes act as proxies)."
)
s_d = (
    f"(d) Retraining without all 77 regex features (581 features): test F1 is {f1_noreg:.4f} "
    f"(recall {ab_noreg['test']['recall']:.4f}, FPR {ab_noreg['test']['fpr']:.4f}), a small but real loss compared to {f1_ctrl:.4f}."
)
s_e = (
    "(e) Validity caveat: edits to the feature table are not real PDF edits, and none of this measures real-world evasion."
)

what_this_shows = '\n\n'.join([s_a, s_b, s_c, s_d, s_e])
assert 'drop' not in s_c.lower() and 'no loss' in s_c.lower()
assert 'falls' in s_a and 'rises' in s_a
print('WHAT THIS SHOWS:\n' + what_this_shows)


WHAT THIS SHOWS:
(a) Shipped model with links removed (/URI = 0 in the feature table): recall at t = 0.4083 falls from 92.73% to 63.45% (paired 95% CI for change: [-30.51%, -28.13%]) and the review rate rises from 3.77% to 16.02%. Among link-bearing malicious files, 1814 lose the malicious verdict: 1619 move to needs_review and 195 move to benign.

(b) Definition distinction: 'recall at t = 0.4083' counts files with calibrated probability >= 0.4083, whereas a 'malicious verdict' requires calibrated probability > 0.550 (above the review band). These are different quantities, so the 1814 malicious-verdict losses and the 29.28-point recall fall at 0.4083 are not the same quantity.

(c) Retraining without /URI (657 features): test F1 is 0.9476 (95% CI [0.9436, 0.9516]) vs 0.9451 (95% CI [0.941, 0.9494]) for the 658-feature control—the intervals overlap, so there is no loss in test F1 (validation F1 is 0.9784 vs 0.9824). This indicates link information is partly redundant with other feature

## PART B. Small Domain-Shift Check (Synthetic + Anonymised User PDFs)

In [8]:
from src.docsentry_extract import DocSentryFeatureExtractor
import src.docsentry_infer as di

ext = DocSentryFeatureExtractor(processed_dir=PROCESSED_DIR)
di.load_artifacts(models_dir=MODELS_DIR, processed_dir=PROCESSED_DIR)

ben_tr = X_tr.loc[y_tr == 0]
def pct_rank(series, val):
    return round(float(np.mean(series <= val) * 100.0), 2)

demo_rows = []
synth_files = sorted((ROOT / 'data/demo_pdfs/synthetic').glob('*.pdf'))
for p in synth_files:
    df_f = ext.extract(p)
    res = di.predict(df_f)
    demo_rows.append({
        'alias': p.name,
        'group': 'synthetic',
        'size_pct_rank': pct_rank(ben_tr['general_size'], float(df_f['general_size'].iloc[0])),
        'numstrings_pct_rank': pct_rank(ben_tr['strings_numstrings'], float(df_f['strings_numstrings'].iloc[0])),
        'uri_pct_rank': pct_rank(ben_tr[uri_col], float(df_f[uri_col].iloc[0])),
        'entropy_pct_rank': pct_rank(ben_tr['general_entropy'], float(df_f['general_entropy'].iloc[0])),
        'calibrated_probability': res['calibrated_probability'],
        'verdict': res['label'],
    })

user_dir = ROOT / 'data/demo_pdfs/user'
if user_dir.exists():
    user_files = sorted(user_dir.glob('*.pdf'))
    for i, p in enumerate(user_files, start=1):
        df_f = ext.extract(p)
        res = di.predict(df_f)
        demo_rows.append({
            'alias': f'user_pdf_{i}.pdf',
            'group': 'user_anonymised',
            'size_pct_rank': pct_rank(ben_tr['general_size'], float(df_f['general_size'].iloc[0])),
            'numstrings_pct_rank': pct_rank(ben_tr['strings_numstrings'], float(df_f['strings_numstrings'].iloc[0])),
            'uri_pct_rank': pct_rank(ben_tr[uri_col], float(df_f[uri_col].iloc[0])),
            'entropy_pct_rank': pct_rank(ben_tr['general_entropy'], float(df_f['general_entropy'].iloc[0])),
            'calibrated_probability': res['calibrated_probability'],
            'verdict': res['label'],
        })

df_domain = pd.DataFrame(demo_rows)
df_domain.to_csv(REPORTS_DIR / 'domain_shift_demo.csv', index=False)
print(df_domain.to_string(index=False))

                         alias           group  size_pct_rank  numstrings_pct_rank  uri_pct_rank  entropy_pct_rank  calibrated_probability      verdict
 synthetic_a_minimal_blank.pdf       synthetic           0.00                 4.27         82.84              0.24                  0.4082 needs_review
 synthetic_b_one_page_text.pdf       synthetic           4.28                 4.29         82.84              0.38                  0.9808    malicious
synthetic_c_20_page_report.pdf       synthetic           4.50                 4.83         82.84              0.23                  0.7826    malicious
       synthetic_d_25_uris.pdf       synthetic           4.80                 5.73         96.10              0.65                  0.9877    malicious
   synthetic_e_embedded_js.pdf       synthetic           4.28                 4.29         82.84              0.42                  0.9953    malicious
     synthetic_f_large_5mb.pdf       synthetic          87.99                98.41      

In [17]:
# Compute domain-shift summary statistics and column note from reports/domain_shift_demo.csv
df_domain = pd.read_csv(REPORTS_DIR / 'domain_shift_demo.csv')

synth_ae = df_domain[df_domain['alias'].str.startswith(('synthetic_a', 'synthetic_b', 'synthetic_c', 'synthetic_d', 'synthetic_e'))]
synth_f = df_domain[df_domain['alias'].str.startswith('synthetic_f')].iloc[0]
user_df = df_domain[df_domain['group'] == 'user_anonymised']

n_user = len(user_df)
n_user_lt25 = int((user_df['size_pct_rank'] < 25).sum())
u_sz_min, u_sz_max = user_df['size_pct_rank'].min(), user_df['size_pct_rank'].max()
u_st_min, u_st_max = user_df['numstrings_pct_rank'].min(), user_df['numstrings_pct_rank'].max()
u_en_min, u_en_max = user_df['entropy_pct_rank'].min(), user_df['entropy_pct_rank'].max()
u_lt25_min = user_df.loc[user_df['size_pct_rank'] < 25, 'size_pct_rank'].min()
u_lt25_max = user_df.loc[user_df['size_pct_rank'] < 25, 'size_pct_rank'].max()

ae_sz_min, ae_sz_max = synth_ae['size_pct_rank'].min(), synth_ae['size_pct_rank'].max()
ae_st_min, ae_st_max = synth_ae['numstrings_pct_rank'].min(), synth_ae['numstrings_pct_rank'].max()
ae_en_min, ae_en_max = synth_ae['entropy_pct_rank'].min(), synth_ae['entropy_pct_rank'].max()

f_sz = synth_f['size_pct_rank']
f_st = synth_f['numstrings_pct_rank']
f_en = synth_f['entropy_pct_rank']

zero_uri_pct = float(df_domain.loc[df_domain['alias'] == 'synthetic_a_minimal_blank.pdf', 'uri_pct_rank'].iloc[0])
plateau_files = df_domain.loc[df_domain['calibrated_probability'].round(4) == 0.4082, 'alias'].tolist()
n_plateau = len(plateau_files)
n_total = len(df_domain)
n_user_benign = int((user_df['verdict'] == 'benign').sum())
n_user_review = int((user_df['verdict'] == 'needs_review').sum())

domain_shift_para = (
    f"Compared against the training benign distribution, the five small synthetic PDFs (`synthetic_a`–`synthetic_e`) "
    f"fall in the low tail of file size ({ae_sz_min:.2f}th–{ae_sz_max:.2f}th percentile), string count "
    f"({ae_st_min:.2f}th–{ae_st_max:.2f}th percentile), and entropy ({ae_en_min:.2f}th–{ae_en_max:.2f}th percentile). "
    f"`synthetic_f_large_5mb.pdf` is not in the low size tail: its 5.5 MB padding places it at the {f_sz:.2f}th size "
    f"and {f_st:.2f}th string-count percentile, while its entropy remains in the low tail ({f_en:.2f}th percentile). "
    f"By contrast, the {n_user} anonymised user PDFs (`user_pdf_1.pdf`–`user_pdf_9.pdf`) are spread across the range "
    f"of the training benign files with none in the extreme tails: size spans {u_sz_min:.2f}th–{u_sz_max:.2f}th percentile "
    f"({n_user_lt25} of {n_user} are below the 25th percentile, from {u_lt25_min:.2f}th to {u_lt25_max:.2f}th), "
    f"string count spans {u_st_min:.2f}th–{u_st_max:.2f}th percentile, and entropy spans {u_en_min:.2f}th–{u_en_max:.2f}th percentile; "
    f"{n_user_benign} are classified `benign` and {n_user_review} (`user_pdf_3.pdf` and `user_pdf_8.pdf`) are routed to `needs_review`. "
    f"Across all {n_total} demo files, {n_plateau} ({', '.join(f'`{x}`' for x in plateau_files)}) sit on the isotonic calibrator's `0.4082` plateau, which is why they receive `needs_review`."
)

column_note = (
    f"**Column note (`uri_pct_rank`):** `uri_pct_rank = {zero_uri_pct:.2f}` is a tie plateau meaning **zero links** "
    f"(`{zero_uri_pct:.2f}%` of training benign files have `/URI = 0`), not the 83rd percentile of link-heavy files. "
    f"Any value above `{zero_uri_pct:.2f}` indicates at least one `/URI` link (`user_pdf_7.pdf` at `88.29`, `user_pdf_8.pdf` at `89.46`, "
    f"`user_pdf_3.pdf` at `92.46`, and `synthetic_d_25_uris.pdf` at `96.10`)."
)

print(domain_shift_para)
print()
print(column_note)


Compared against the training benign distribution, the five small synthetic PDFs (`synthetic_a`–`synthetic_e`) fall in the low tail of file size (0.00th–4.80th percentile), string count (4.27th–5.73th percentile), and entropy (0.23th–0.65th percentile). `synthetic_f_large_5mb.pdf` is not in the low size tail: its 5.5 MB padding places it at the 87.99th size and 98.41th string-count percentile, while its entropy remains in the low tail (0.26th percentile). By contrast, the 9 anonymised user PDFs (`user_pdf_1.pdf`–`user_pdf_9.pdf`) are spread across the range of the training benign files with none in the extreme tails: size spans 9.20th–74.29th percentile (5 of 9 are below the 25th percentile, from 9.20th to 23.37th), string count spans 15.10th–73.66th percentile, and entropy spans 11.24th–91.53th percentile; 7 are classified `benign` and 2 (`user_pdf_3.pdf` and `user_pdf_8.pdf`) are routed to `needs_review`. Across all 15 demo files, 3 (`synthetic_a_minimal_blank.pdf`, `synthetic_f_larg

### Domain-Shift Interpretation

Compared against the training benign distribution, the five small synthetic PDFs (`synthetic_a`–`synthetic_e`) fall in the low tail of file size (0.00th–4.80th percentile), string count (4.27th–5.73th percentile), and entropy (0.23th–0.65th percentile). `synthetic_f_large_5mb.pdf` is not in the low size tail: its 5.5 MB padding places it at the 87.99th size and 98.41th string-count percentile, while its entropy remains in the low tail (0.26th percentile). By contrast, the 9 anonymised user PDFs (`user_pdf_1.pdf`–`user_pdf_9.pdf`) are spread across the range of the training benign files with none in the extreme tails: size spans 9.20th–74.29th percentile (5 of 9 are below the 25th percentile, from 9.20th to 23.37th), string count spans 15.10th–73.66th percentile, and entropy spans 11.24th–91.53th percentile; 7 are classified `benign` and 2 (`user_pdf_3.pdf` and `user_pdf_8.pdf`) are routed to `needs_review`. Across all 15 demo files, 3 (`synthetic_a_minimal_blank.pdf`, `synthetic_f_large_5mb.pdf`, `user_pdf_8.pdf`) sit on the isotonic calibrator's `0.4082` plateau, which is why they receive `needs_review`.

**Column note (`uri_pct_rank`):** `uri_pct_rank = 82.84` is a tie plateau meaning **zero links** (`82.84%` of training benign files have `/URI = 0`), not the 83rd percentile of link-heavy files. Any value above `82.84` indicates at least one `/URI` link (`user_pdf_7.pdf` at `88.29`, `user_pdf_8.pdf` at `89.46`, `user_pdf_3.pdf` at `92.46`, and `synthetic_d_25_uris.pdf` at `96.10`).
